In [70]:
import matplotlib.pyplot as plt
import numpy as np
import os
import random


from paths import *
from reshandler import DictResHandler

In [41]:
def read_result_at(res_save_dir, epoch): 
    all_handler = DictResHandler(whole_res_dir=res_save_dir, 
                                 file_prefix=f"all-{epoch}")

    all_handler.read()

    return all_handler.res


def plot_attention_matrix(attention_matrix, 
                          query_labels=None, key_labels=None, 
                          sep_frame1=None, sep_frame2=None, 
                          title="Attention Pattern", weight_range=(0, 1), 
                          save_path=None):
    """
    Plots the attention matrix with queries on the x-axis and keys on the y-axis,
    and adds dotted lines at specified frame positions.

    Parameters:
        attention_matrix (np.ndarray): The attention matrix of shape (num_queries, num_keys).
        query_labels (list of str): Labels for the queries (x-axis). Default is None.
        key_labels (list of str): Labels for the keys (y-axis). Default is None.
        sep_frame1 (int): Position of the first dotted line. Default is None.
        sep_frame2 (int): Position of the second dotted line. Default is None.
        title (str): Title of the plot. Default is "Attention Pattern".
    """
    vmin, vmax = weight_range

    plt.figure(figsize=(10, 8))
    plt.imshow(attention_matrix, cmap='viridis', aspect='auto', vmin=vmin, vmax=vmax)
    plt.colorbar(label="Attention Weight")
    
    # Add dotted lines if sep_frame1 and sep_frame2 are provided
    if sep_frame1 is not None:
        plt.axvline(x=sep_frame1, color='white', linestyle='--', linewidth=1)  # Dotted line on x-axis
        plt.axhline(y=sep_frame1, color='white', linestyle='--', linewidth=1)  # Dotted line on y-axis
    if sep_frame2 is not None:
        plt.axvline(x=sep_frame2, color='white', linestyle='--', linewidth=1)  # Dotted line on x-axis
        plt.axhline(y=sep_frame2, color='white', linestyle='--', linewidth=1)  # Dotted line on y-axis

    if query_labels is not None:
        plt.xticks(ticks=np.arange(len(query_labels)), labels=query_labels, rotation=90)
    if key_labels is not None:
        plt.yticks(ticks=np.arange(len(key_labels)), labels=key_labels)
    
    plt.xlabel("Queries (Decoding Frames)")
    plt.ylabel("Keys (Encoder Output Frames)")
    plt.title(title)
    plt.tight_layout()
    # plt.show()
    if save_path is not None:
        plt.savefig(save_path)
    else:
        plt.show()
    plt.close()

In [80]:
train_name = "E_0A"
ts = "1113024340"
eval_dir = os.path.join(model_save_, f"eval-{train_name}-{ts}")

In [93]:
dim = 8
balance_type = "b"
run_number = random.randint(1, 5)
target_res_dir = os.path.join(eval_dir, f"recon{dim}-phi", f"{balance_type}", f"{run_number}")
output_dir = os.path.join(eval_dir, f"recon{dim}-phi", f"{balance_type}", "attention_plots")
mk(output_dir)

# random idx to sample
idx = random.randint(0, 900)
outoutput_dir = os.path.join(output_dir, f"run-{run_number}-sample-{idx}")
mk(outoutput_dir)
print(run_number, idx)

for epoch in range(101): # 0~100
    allres = read_result_at(target_res_dir, epoch)
    attention_matrix = allres["attn"][idx]
    first_sep_frame = allres["sep-frame1"][idx]
    second_sep_frame = allres["sep-frame2"][idx]
    v1_name = allres["v1-name"][idx]
    s_name = allres["sn"][idx]
    v2_name = allres["vn"][idx]

    query_tokens = [v1_name] * first_sep_frame + [s_name] * (second_sep_frame - first_sep_frame) + [v2_name] * (len(attention_matrix) - second_sep_frame)
    key_tokens = query_tokens
    plot_attention_matrix(attention_matrix, query_labels=query_tokens, key_labels=key_tokens, 
                          sep_frame1=first_sep_frame, sep_frame2=second_sep_frame,
                          title=f"Attention: {v1_name}-{s_name}-{v2_name} at Epoch {epoch}", 
                          weight_range=(0, 0.3), 
                          save_path=os.path.join(outoutput_dir, f"attention-epoch-{epoch}.png"))

691
